In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import os
from scipy.signal import find_peaks

plt.rcParams.update({
    "text.usetex": True,                
    "font.family": "serif",
    "font.serif": ["Computer Modern"],
    "figure.dpi": 150,                   
    "grid.alpha": 0.4,                    
})


In [3]:
# Physics Constants for Compactness
SOLAR_MASS_KG = 1.98847e30
G = 6.67430e-11
c = 299792458
COMPACTNESS_FACTOR = G / c**2

def process_stability_for_dataset(csv_path, clean_save_name=None):
    """
    Cleans the raw parallel dataset, calculates compactness, computes 
    the 2D stability grid using the Jacobian eigenvalue criterion, 
    and saves the complete dataset with an 'is_stable' boolean column.
    """
    # 1. Load and Clean
    df = pd.read_csv(csv_path)
    original_len = len(df)
    
    # ==========================================
    # Cutoff for maximum QM central pressure
    # ==========================================
    df = df[df['p_c_qm'] <= 1e4]
    
    df = df.drop_duplicates(subset=['p_c_qm', 'p_c_dm'], keep='last')
    
    # Sort values so the grid pivots perfectly smoothly
    df = df.sort_values(by=['p_c_dm', 'p_c_qm'])
    print(f"[{os.path.basename(csv_path)}] Filtered & Cleaned {original_len - len(df)} rows. Final size: {len(df)}")
    
    # 2. Add Compactness
    df['compactness'] = (df['M_total_sol'] * SOLAR_MASS_KG) / (df['R_total_km'] * 1000) * COMPACTNESS_FACTOR

    # 3. Extract Grid Axes
    p_c_DM = np.unique(df['p_c_dm'].values)
    p_c_QM = np.unique(df['p_c_qm'].values)

    # 4. Pivot the Data
    grid_N_qm = df.pivot(index='p_c_dm', columns='p_c_qm', values='N_quark').values
    grid_N_dm = df.pivot(index='p_c_dm', columns='p_c_qm', values='N_dm').values
    mass_grid = df.pivot(index='p_c_dm', columns='p_c_qm', values='M_total_sol')
    compact_grid = df.pivot(index='p_c_dm', columns='p_c_qm', values='compactness')

    # 5. Stability Math (Jacobian Matrix Eigenvalues)
    # Calculem les derivades indicant explícitament l'espaiat de les coordenades
    # Això millora la precisió si les graelles no són totalment lineals
    dNqm_dPdm, dNqm_dPqm = np.gradient(grid_N_qm, p_c_DM, p_c_QM)
    dNdm_dPdm, dNdm_dPqm = np.gradient(grid_N_dm, p_c_DM, p_c_QM)

    rows, cols = grid_N_qm.shape
    stability_grid = np.zeros((rows, cols), dtype=bool)

    # ==========================================
    # MODIFICAT: Iteració amb Propagació 2D (Flood-fill)
    # ==========================================
    for i in range(rows):
        for j in range(cols):
            # CONDICIÓ DE CONNECTIVITAT 2D:
            # Llevat del punt d'origen (0,0), exigim que l'estabilitat provingui
            # de pressions inferiors, ja sigui per QM (j-1) o per DM (i-1).
            # Si tots els camins de pressió inferior han col·lapsat, aquest punt 
            # no pot renéixer, per molt que els eigenvalors siguin positius.
            
            te_vei_estable = False
            
            if i == 0 and j == 0:
                # El primer punt de tots s'assumeix que prové del buit i té opció a ser estable
                te_vei_estable = True 
            else:
                vei_esquerra_estable = stability_grid[i, j-1] if j > 0 else False
                vei_sot_estable = stability_grid[i-1, j] if i > 0 else False
                
                if vei_esquerra_estable or vei_sot_estable:
                    te_vei_estable = True

            # Si està desconnectat de la regió estable principal, col·lapsa directament
            if not te_vei_estable:
                stability_grid[i, j] = False
                continue

            # Si està connectat, aleshores i només aleshores, mirem la matemàtica del Jacobià
            matrix = np.array([
                [dNqm_dPqm[i, j], dNqm_dPdm[i, j]],
                [dNdm_dPqm[i, j], dNdm_dPdm[i, j]]
            ])
            
            # Check for NaNs before calculating Eigenvalues
            if not np.isfinite(matrix).all():
                stability_grid[i, j] = False
                continue  
            
            eigenvalues = np.linalg.eigvals(matrix)
            kappa_A, kappa_B = np.real(eigenvalues[0]), np.real(eigenvalues[1])
            
            # El criteri físic del Jacobià:
            if kappa_A > 0 and kappa_B > 0:
                stability_grid[i, j] = True
            else:
                stability_grid[i, j] = False

    # 6. Map the 2D Stability Grid back to the 1D DataFrame
    stab_df = pd.DataFrame(stability_grid, index=mass_grid.index, columns=mass_grid.columns)
    
    # 'Stack' flattens it back into a 1D list, and we reset the index to get the pressure columns back
    stab_stacked = stab_df.stack().reset_index()
    stab_stacked.columns = ['p_c_dm', 'p_c_qm', 'is_stable']
    
    # Merge the 'is_stable' column into the main dataframe
    df = pd.merge(df, stab_stacked, on=['p_c_dm', 'p_c_qm'], how='left')

    # 7. Save the final dataframe with the new 'is_stable' column
    if clean_save_name:
        save_path = f'../data/mr_library_twofluids/cleaned/{clean_save_name}.csv'
        df.to_csv(save_path, index=False)
        print(f"Saved data with 'is_stable' column to: {save_path}")

    # Package everything needed for plotting
    return {
        'df': df,
        'p_c_QM': p_c_QM,
        'p_c_DM': p_c_DM,
        'mass_grid': mass_grid,
        'compact_grid': compact_grid,
        'stability_grid': stability_grid
    }

In [ ]:

datasets_to_analyze = {
    "../data/mr_library_twofluids/mr_mb_200_n_4.csv": {
        "title": r"Quark + Bosonic ($m_b = 200$ MeV, $n = 4$)",
        "save_prefix": "QM_BDM_mb200_n4"
    },
    "../data/mr_library_twofluids/mr_mb_300_n_4.csv": {
        "title": r"Quark + Bosonic ($m_b = 300$ MeV, $n = 4$)",
        "save_prefix": "QM_BDM_mb300_n4"
    },
    "../data/mr_library_twofluids/mr_mb_500_n_4.csv": {
        "title": r"Quark + Bosonic ($m_b = 500$ MeV, $n = 4$)",
        "save_prefix": "QM_BDM_mb500_n4"
    },
    "../data/mr_library_twofluids/mr_mb_1000_n_4.csv": {
        "title": r"Quark + Bosonic ($m_b = 1000$ MeV, $n = 4$)",
        "save_prefix": "QM_BDM_mb1000_n4"
    },
    "../data/mr_library_twofluids/mr_mb_200_n_40.csv": {
        "title": r"Quark + Bosonic ($m_b = 200$ MeV, $n = 40$)",
        "save_prefix": "QM_BDM_mb200_n40"
    },
    "../data/mr_library_twofluids/mr_mb_300_n_40.csv": {
        "title": r"Quark + Bosonic ($m_b = 300$ MeV, $n = 40$)",
        "save_prefix": "QM_BDM_mb300_n40"
    },
    "../data/mr_library_twofluids/mr_mb_500_n_40.csv": {
        "title": r"Quark + Bosonic ($m_b = 500$ MeV, $n = 40$)",
        "save_prefix": "QM_BDM_mb500_n40"
    },
    "../data/mr_library_twofluids/mr_mb_1000_n_40.csv": {
        "title": r"Quark + Bosonic ($m_b = 1000$ MeV, $n = 40$)",
        "save_prefix": "QM_BDM_mb1000_n40"
    }
}

# Run the entire pipeline!
for filepath, config in datasets_to_analyze.items():
    print(f"\n================ Processing: {config['title']} ================")
    
    # 1. Clean data and compute stability
    data_dict = process_stability_for_dataset(
        csv_path=filepath, 
        clean_save_name=config['save_prefix'] + "_Cleaned"
    )
    


================ Processing: Quark + Bosonic ($m_b = 200$ MeV, $n = 4$) ================
[mr_mb_200_n_4.csv] Filtered & Cleaned 8700 rows. Final size: 81000
Saved data with 'is_stable' column to: ../data/mr_library_twofluids/cleaned/QM_BDM_mb200_n4_Cleaned.csv

================ Processing: Quark + Bosonic ($m_b = 300$ MeV, $n = 4$) ================
[mr_mb_300_n_4.csv] Filtered & Cleaned 8700 rows. Final size: 81000
Saved data with 'is_stable' column to: ../data/mr_library_twofluids/cleaned/QM_BDM_mb300_n4_Cleaned.csv

================ Processing: Quark + Bosonic ($m_b = 500$ MeV, $n = 4$) ================
[mr_mb_500_n_4.csv] Filtered & Cleaned 8700 rows. Final size: 81000
Saved data with 'is_stable' column to: ../data/mr_library_twofluids/cleaned/QM_BDM_mb500_n4_Cleaned.csv

================ Processing: Quark + Bosonic ($m_b = 1000$ MeV, $n = 4$) ================
[mr_mb_1000_n_4.csv] Filtered & Cleaned 8700 rows. Final size: 81000
Saved data with 'is_stable' column to: ../data/mr_libr